In [ ]:
import sys
import os
from typing import NamedTuple
import math
import numpy as np
import scipy as sp
import pandas as pd
from matplotlib import pyplot as plt
import pickle

from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold
import jax


sys.path.append(os.path.join(os.getcwd(), '../'))
from utils_fly import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils
from models import bom_rs as bom
from models import hgm_rs as hgm


jax.config.update("jax_enable_x64", True)

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)
importlib.reload(bom)
importlib.reload(hgm)

# Functions

In [ ]:
def seperate_predictions(file_ids, columns, models_spn_man_rec_res_pres_resss_bss_art):
	models_spn = [
		[
			pd.DataFrame(data=np.array(split[0])[None], columns=columns[i]) for split in models_spn_man_rec_res_pres_resss_bss_art[i]
		] for i in range(len(file_ids))
	]
	models_man = [
		[
			pd.DataFrame(data=np.array(split[1]), columns=[f'M{j}' for j in range(np.array(split[1]).shape[1])]) for split in models_spn_man_rec_res_pres_resss_bss_art[i]
		] for i in range(len(file_ids))
	]
	(models_rec,
	 models_res,
	 models_pres) = tuple(
		[
			[
				pd.DataFrame(data=np.array(split[m]), columns=columns[i]) for split in models_spn_man_rec_res_pres_resss_bss_art[i]
			] for i in range(len(file_ids))
		] for m in [2, 3, 4]
	)
	(models_resss,
	 models_bss) = tuple(
		[
			[
				pd.DataFrame(data=np.array(split[m]), columns=[f'SS{j}' for j in range(np.array(split[m]).shape[1])]) for split in models_spn_man_rec_res_pres_resss_bss_art[i]
			] for i in range(len(file_ids))
		] for m in [5, 6]
	)
	return models_spn, models_man, models_rec, models_res, models_pres, models_resss, models_bss

In [ ]:
def assemble(boms_scores_per_neuron, hgms_scores_per_neuron):
	neurons_labels = [scores_per_neuron[0].columns for scores_per_neuron in hgms_scores_per_neuron]

	boms_scores_per_neuron = [np.array([split.values[0] for split in scores]) for scores in boms_scores_per_neuron]
	hgms_scores_per_neuron = [np.array([split.values[0] for split in scores]) for scores in hgms_scores_per_neuron]

	neuron_scores = dict()
	boms_scores_per_neuron_avg = [np.mean(scores, axis=0) for scores in boms_scores_per_neuron]
	hgms_scores_per_neuron_avg = [np.mean(scores, axis=0) for scores in hgms_scores_per_neuron]
	for label_list, score_bom_list, score_hgm_list in zip(neurons_labels, boms_scores_per_neuron_avg, hgms_scores_per_neuron_avg):
		sort_indices = np.argsort(score_hgm_list)[::-1]
		label_list = label_list[sort_indices]
		score_bom_list = score_bom_list[sort_indices]
		score_hgm_list = score_hgm_list[sort_indices]

		for i in range(len(label_list)):
			label = label_list[i]
			if label != '' and label[-1] != '_':
				score_bom_new, score_hgm_new = np.array(score_bom_list[i]), np.array(score_hgm_list[i])
				if len(score_hgm_new.shape) == 0:
					score_bom_new = score_bom_new.reshape(1, 1)
					score_hgm_new = score_hgm_new.reshape(1, 1)
				if label not in neuron_scores.keys():
					neuron_scores[label] = (score_bom_new, score_hgm_new)
				else:
					score_bom_pstill, score_hgm_pstill = neuron_scores[label]
					neuron_scores[label] = (np.vstack([score_bom_pstill, score_bom_new]), np.vstack([score_hgm_pstill, score_hgm_new]))

	return neuron_scores

# read & prep

In [ ]:
default_hyperparameters = {
	'dimensions': 2,
	'loss_man_lambda': 0.0,
	'loss_res_lambda': 1.0 - 0.0,
	'vb_reg_lambda': 0.0,
	'wb_reg_lambda': 0.0,
	'wm_reg_lambda': 0.0,
}

ll = f'll{int(default_hyperparameters['loss_man_lambda'] * 100):02d}'

with open('../sweep/hyperparams.pkl', 'rb') as f:
    swept_hyperparameters = pickle.load(f)

bom_reg_lambdas = {
	# 'vb_reg_lambda': default_hyperparameters['vb_reg_lambda'],
	'vb_reg_lambda': swept_hyperparameters[ll]['bom_rs']['means']['vb_reg_lambda']['geometric'],
	# 'wb_reg_lambda': default_hyperparameters['wb_reg_lambda'],
	'wb_reg_lambda': swept_hyperparameters[ll]['bom_rs']['means']['wb_reg_lambda']['geometric'],
}
hgm_reg_lambdas = {
	# 'vb_reg_lambda': default_hyperparameters['vb_reg_lambda'],
	'vb_reg_lambda': swept_hyperparameters[ll]['hgm_rs']['means']['vb_reg_lambda']['geometric'],
	# 'wb_reg_lambda': default_hyperparameters['wb_reg_lambda'],
	'wb_reg_lambda': swept_hyperparameters[ll]['hgm_rs']['means']['wb_reg_lambda']['geometric'],
	# 'wm_reg_lambda': default_hyperparameters['wm_reg_lambda'],
	'wm_reg_lambda': swept_hyperparameters[ll]['hgm_rs']['means']['wm_reg_lambda']['geometric'],
}

In [ ]:
read_ids = list(range(0, 12+1))
# read_ids = [0,	# paper
# 			1,	# nice states
# 			2,	# nice manifold
# 			5,	# variety
# 			6,	# nice quiescence
# 			7,	# nice states
# 			9,	# nice heatmap
# 			11]	# variety
# read_ids = [0]
plot_id = 0

In [ ]:
# read

path = os.path.join(read_utils.get_data_path(), 'brain')

(file_ids, filenames,
 traces,
 states, markers) = read_utils.get_data(path, read_ids)

# # clear nan values
# for i in range(len(file_ids)):
# 	mask = markers[i].notna().all(axis=1)
#
# 	traces[i] = traces[i][mask]
# 	markers[i] = markers[i][mask]
# 	states[i] = states[i][mask]
#
# 	#filenames[i] = filenames[i][mask]

# # remove unused discrete state labels
# states = [ds[selection_bhl] for ds in states]

# naive ID unknown neurons
for i in range(len(file_ids)):
	traces[i].columns = pd.Index([
		f'{file_ids[i]:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces[i].columns, traces[i].columns == ''))
	])

In [ ]:
# prep

def highpass_butter(x, fs, cutoff_hz, order=4, axis=0):
    return sp.signal.sosfiltfilt(sp.signal.butter(order, cutoff_hz, btype='highpass', fs=fs, output='sos'), x, axis=axis)

sampling_rates = [read_utils.get_sampling_rate(os.path.join(path, filenames[file_ids.index(file_id)])) for file_id in file_ids]

traces = [pd.DataFrame(highpass_butter(trace.values, sr, 1 / 100), columns=trace.columns) for trace, sr in zip(traces, sampling_rates)]

markers_grad = [pd.DataFrame(np.gradient(marker.values, axis=0), columns=marker.columns) for marker in markers]

In [ ]:
limit_n = 5
selection_bhl = ['still', 'walk', 'front-groom', 'back-groom', 'abdomen-move']

plot_idx = file_ids.index(plot_id)

# states = [ds[selection_bhl] for ds in states]

# read models

In [ ]:
organism = 'fly-2023-schaffer'

bom_load_name = os.path.join(organism, ll, f'bom_wb-{bom_reg_lambdas['wb_reg_lambda']:.2E}')
hgm_load_name = os.path.join(organism, ll, f'hgm_wb-{hgm_reg_lambdas['wb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}')

In [ ]:
boms_params = train_utils.load_params(bom_load_name, filenames)

boms_losses = train_utils.load_losses(bom_load_name, filenames)
boms_losses_train = [losses[0] for losses in boms_losses]
boms_man_losses_train = [losses[1] for losses in boms_losses]
boms_res_losses_train = [losses[2] for losses in boms_losses]
boms_mpe_losses_train = [losses[3] for losses in boms_losses]
boms_losses_test = [losses[4] for losses in boms_losses]
boms_man_losses_test = [losses[5] for losses in boms_losses]
boms_res_losses_test = [losses[6] for losses in boms_losses]
boms_mpe_losses_test = [losses[7] for losses in boms_losses]

hgms_params = train_utils.load_params(hgm_load_name, filenames)

hgms_losses = train_utils.load_losses(hgm_load_name, filenames)
hgms_losses_train = [losses[0] for losses in hgms_losses]
hgms_man_losses_train = [losses[1] for losses in hgms_losses]
hgms_res_losses_train = [losses[2] for losses in hgms_losses]
hgms_mpe_losses_train = [losses[3] for losses in hgms_losses]
hgms_losses_test = [losses[4] for losses in hgms_losses]
hgms_man_losses_test = [losses[5] for losses in hgms_losses]
hgms_res_losses_test = [losses[6] for losses in hgms_losses]
hgms_mpe_losses_test = [losses[7] for losses in hgms_losses]

 # gcamp

### losses

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_losses_train, boms_losses_train,
							  title='gcamp hgm vs bom: train',
							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_mpe_losses_train, boms_mpe_losses_train,
							  title='gcamp hgm vs bom: mpe train',
							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_losses_test, boms_losses_test,
							  title='gcamp hgm vs bom: test',
							  ylimits=None)

In [ ]:
plot_utils.plot_losses_splits(filenames,
							  hgms_mpe_losses_test, boms_mpe_losses_test,
							  title='gcamp hgm vs bom: mpe test',
							  ylimits=None)

### predict & score

In [ ]:
splits = [list(StratifiedKFold(n_splits=5, shuffle=False).split(np.zeros(traces[i].shape[0]), states[i][['still']].values)) for i in range(len(file_ids))]

In [ ]:
boms_spn_man_rec_res_pres_resss_bss = [[
	bom.score_fn(params, traces[i].values[eval], markers[i].values[eval]) for params, (train, eval) in zip(boms_params[i], splits[i])
] for i in range(len(file_ids))]

hgms_spn_man_rec_res_pres_resss_bss = [[
	hgm.score_fn(params, traces[i].values[eval], markers[i].values[eval]) for params, (train, eval) in zip(hgms_params[i], splits[i])
] for i in range(len(file_ids))]

(boms_spn,
 boms_man,
 boms_rec,
 boms_res,
 boms_pres,
 boms_resss,
 boms_bss) = seperate_predictions(file_ids, [trace.columns for trace in traces], boms_spn_man_rec_res_pres_resss_bss)

(hgms_spn,
 hgms_man,
 hgms_rec,
 hgms_res,
 hgms_pres,
 hgms_resss,
 hgms_bss) = seperate_predictions(file_ids, [trace.columns for trace in traces], hgms_spn_man_rec_res_pres_resss_bss)

### performance

In [ ]:
neuron_scores = assemble(boms_spn, hgms_spn)

In [ ]:
plot_utils.plot_performance_plot(
	neuron_scores, None,
	title='scores for all neurons', limits=(-0.55, 0.55, -0.55, 0.55), force_medians=True
)

### manifold

In [ ]:
boms_spn_man_rec_res_pres_resss_bss = [[
	bom.score_fn(params, traces[i].values, markers[i].values) for params in boms_params[i]
] for i in range(len(file_ids))]

hgms_spn_man_rec_res_pres_resss_bss = [[
	hgm.score_fn(params, traces[i].values, markers[i].values) for params in hgms_params[i]
] for i in range(len(file_ids))]

(boms_spn_full,
 boms_man_full,
 boms_rec_full,
 boms_res_full,
 boms_pres_full,
 boms_resss_full,
 boms_bss_full) = seperate_predictions(file_ids, [trace.columns for trace in traces], boms_spn_man_rec_res_pres_resss_bss)

(hgms_spn_full,
 hgms_man_full,
 hgms_rec_full,
 hgms_res_full,
 hgms_pres_full,
 hgms_resss_full,
 hgms_bss_full) = seperate_predictions(file_ids, [trace.columns for trace in traces], hgms_spn_man_rec_res_pres_resss_bss)

In [ ]:
if default_hyperparameters['dimensions'] > 1:
	splits = 5
	plot_utils.plot_manifolds([f'{str(plot_id)}: {filenames[plot_idx]} s{s}' for s in range(splits)],
							  [hgms_man_full[plot_idx][s] for s in range(splits)],
							  [states[plot_idx] for s in range(splits)],
							  None, None,
							  title='gcamp manifolds', plots_per_row=splits, top_space=3.5,
							  save_path=None)

### traces


In [ ]:
_, _, traces_cca_rec, _, _, _, _, _ = train_utils.get_manifolds_cca(traces, states, n_cv=default_hyperparameters['dimensions'], n_pc=0)
traces_cca_res = [pd.DataFrame(data=trace.values - trace_cca_rec.values, columns=trace.columns) for trace, trace_cca_rec in zip(traces, traces_cca_rec)]

In [ ]:
plot_n = 5
best_n_hgm = [item[0] for item in neuron_scores.items()][:plot_n]

In [ ]:
splits = 5
plot_utils.plot_traces_splits(
	[hgms_rec_full[plot_idx][i][best_n_hgm] for i in range(splits)],
	[boms_rec_full[plot_idx][i][best_n_hgm] for i in range(splits)],
	traces_cca_rec[plot_idx][best_n_hgm],
	states[plot_idx][['still']], label='quiescent',
	title=f'{filenames[plot_idx]} gcamp hgm & bom definitions manifold', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_MAN'

In [ ]:
splits = 5
plot_utils.plot_traces_splits(
	[hgms_res_full[plot_idx][i][best_n_hgm] for i in range(splits)],
	[boms_res_full[plot_idx][i][best_n_hgm] for i in range(splits)],
	traces_cca_res[plot_idx][best_n_hgm],
	states[plot_idx][['still']], label='quiescent',
	title=f'{filenames[plot_idx]} gcamp hgm & bom definitions residuals', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_MAN'

In [ ]:
splits = 5
for i in range(splits):
	plot_utils.plot_traces_splits_pairs(
		[hgms_res_full[plot_idx][i][best_n_hgm],
		 hgms_pres_full[plot_idx][i][best_n_hgm]],
		[boms_res_full[plot_idx][i][best_n_hgm],
		 boms_pres_full[plot_idx][i][best_n_hgm]],
		traces_cca_res[plot_idx][best_n_hgm],
		states[plot_idx][['still']], label='quiescent',
		title=f'{filenames[plot_idx]} split {i} gcamp hgm & bom predictions residuals', ylimits=None, plots_per_row=1,
		save_path=None)  #../../out/plots/f'GCAMP_PRES_S{i}'

In [ ]:
# if gcamp:
# 	splits = 5
# 	plot_utils.plot_traces_splits(
# 		[hgms_rec_full[plot_idx][i][selection_n_traces] for i in range(splits)],
# 		[boms_rec_full[plot_idx][i][selection_n_traces] for i in range(splits)],
# 		traces_cca_rec[plot_idx][selection_n_traces],
# 		states[plot_idx][['still']], label='quiescent',
# 		title=f'{filenames[plot_idx]} gcamp hgm & bom definitions manifold', ylimits=None, plots_per_row=1,
# 		save_path=None)  #'../../out/plots/GCAMP_MAN'

In [ ]:
# if gcamp:
# 	splits = 5
# 	plot_utils.plot_traces_splits(
# 		[hgms_res_full[plot_idx][i][selection_n_traces] for i in range(splits)],
# 		[boms_res_full[plot_idx][i][selection_n_traces] for i in range(splits)],
# 		traces_cca_res[plot_idx][selection_n_traces],
# 		states[plot_idx][['still']], label='quiescent',
# 		title=f'{filenames[plot_idx]} gcamp hgm & bom definitions residuals', ylimits=None, plots_per_row=1,
# 		save_path=None)  #'../../out/plots/GCAMP_MAN'

In [ ]:
# if gcamp:
# 	splits = 5
# 	for i in range(splits):
# 		plot_utils.plot_traces_splits_pairs(
# 			[hgms_res_full[plot_idx][i][selection_n_traces],
# 			 hgms_pres_full[plot_idx][i][selection_n_traces]],
# 			[boms_res_full[plot_idx][i][selection_n_traces],
# 			 boms_pres_full[plot_idx][i][selection_n_traces]],
# 			traces_cca_res[plot_idx][selection_n_traces],
# 			states[plot_idx][['still']], label='quiescent',
# 			title=f'{filenames[plot_idx]} split {i} gcamp hgm & bom predictions residuals', ylimits=None, plots_per_row=1,
# 			save_path=None)  #../../out/plots/f'GCAMP_PRES_S{i}'

In [ ]:
splits = 5
plot_utils.plot_traces_splits(
	[hgms_bss_full[plot_idx][i] for i in range(splits)][0:1],
	[boms_bss_full[plot_idx][i] for i in range(splits)][0:1],
	None,
	states[plot_idx][['still']], label='quiescent',
	title=f'{filenames[plot_idx]} s0 gcamp hgm & bom behavioral subspace', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_BSS'

In [ ]:
splits = 5
plot_utils.plot_traces_splits(
	[hgms_resss_full[plot_idx][i] for i in range(splits)][0:1],
	[boms_resss_full[plot_idx][i] for i in range(splits)][0:1],
	None,
	states[plot_idx][['still']], label='quiescent',
	title=f'{filenames[plot_idx]} s0 gcamp hgm & bom residual subspace', ylimits=None, plots_per_row=1,
	save_path=None)  #'../../out/plots/GCAMP_RESSS'

# sanity check

In [ ]:
idx_SC = 0
split_SC = 0

VB_SC = boms_params[idx_SC][split_SC]['model']['VB']
B_ss_SC = boms_bss_full[idx_SC][split_SC].values
N_res_pred_SC = boms_pres_full[idx_SC][split_SC].values

In [ ]:
# SANITY CHECK: The predicted residual subspace should be equal to the behavioral subspace

def get_reconstruction_matrix(Y, X):
	# best reconstruction matrix that reconstruct X using Y
	# least-squares reconstruction (NOT Y @ jax.numpy.linalg.pinv(UN))
	# return jnp.linalg.inv(Y.T @ Y) @ Y.T @ X
	# return jnp.linalg.solve(Y.T @ Y, Y.T @ X)
	return np.linalg.lstsq(Y, X, rcond=None)[0]

N_res_pred_ss_SC = N_res_pred_SC @ get_reconstruction_matrix(N_res_pred_SC, B_ss_SC)

print(B_ss_SC - N_res_pred_ss_SC)

In [ ]:
# ALSO INTERESTING: The least-squares solution to reconstruct B_ss is basically equal to pseudo-inverting VB:

N_res_pred_ss_pinv_SC = N_res_pred_SC @ np.linalg.pinv(VB_SC)

print(N_res_pred_ss_SC - N_res_pred_ss_pinv_SC)

# relevant neurons

In [ ]:
relevant_hgm_n = {key: value for key, value in neuron_scores.items() if value[0] > 0 and value[1] > 0}

In [ ]:
plot_utils.plot_performance_plot(
	{key: neuron_scores[key] for key in relevant_hgm_n}, None,
	title='scores for relevant neurons', limits=(-0.55, 0.55, -0.55, 0.55), force_medians=True
)

In [ ]:
plot_utils.plot_improvement_plot(
	{key: neuron_scores[key] for key in relevant_hgm_n}, None,
	title='improvements for relevant neurons', limits=(-0.25, 0.25, -0.25, 0.25), force_medians=True
)